# Efficient Reasoning Transfer - calibrated Colab launcher
Run one stage at a time and inspect each decision file. Budget and gamma are selected only from development data; this notebook does not launch main/test runs automatically.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/efficient-reasoning-transfer'  # change if needed
%cd $PROJECT

In [ ]:
!python --version
!nvidia-smi
!df -h /content /content/drive/MyDrive
!python -m pip install --upgrade pip
!python -m pip install -e '.[train,dev]'

## Repository checks and deterministic pilot data

In [ ]:
from pathlib import Path
!ruff check .
!pytest -m 'not gpu'
pilot_manifest = Path('data/generated/pilot/manifest.json')
if not pilot_manifest.exists():
    !python -m effreason.data build --config configs/pilot.yaml
else:
    print('Pilot data already exists; auditing without overwriting it.')
!python -m effreason.data audit --manifest data/generated/pilot/manifest.json

## Stage 1 - completion-budget calibration
Collect four samples for each development item (72 responses with the pilot data). The initial 4,096-token cap is a probe, not the final choice.

In [ ]:
!python -m effreason.evaluate --config configs/pilot.yaml --condition initial_budget_probe --split dev --samples-per-example 4 --output runs/initial_budget_probe

In [ ]:
!mkdir -p analysis/calibration
!python -m effreason.calibrate budget --run-dir runs/initial_budget_probe --output analysis/calibration/budget_decision.json --candidates 512 1024 2048 4096 8192 --quantile 0.95 --headroom 1.10 --max-truncation-rate 0.02 --min-samples 50 --min-correct-samples 20

In [ ]:
import json
with open('analysis/calibration/budget_decision.json', encoding='utf-8') as stream:
    budget_decision = json.load(stream)
display(budget_decision)
if budget_decision['status'] != 'selected':
    raise RuntimeError('Budget not selected. Follow the reported status; a larger probe or more samples is required.')
SELECTED_B = int(budget_decision['recommended_budget'])
print('Selected completion budget:', SELECTED_B)

## Validate the selected budget before training

In [ ]:
!python -m effreason.evaluate --config configs/pilot.yaml --condition initial_calibrated --split dev --samples-per-example 2 --completion-budget $SELECTED_B --output runs/initial_calibrated_dev

In [ ]:
with open('runs/initial_calibrated_dev/predictions.jsonl', encoding='utf-8') as stream:
    validation_rows = [json.loads(line) for line in stream if line.strip()]
validation_truncation_rate = sum(bool(row['truncated']) for row in validation_rows) / len(validation_rows)
print('Validation truncation rate:', validation_truncation_rate)
if validation_truncation_rate > 0.02:
    raise RuntimeError('Selected budget truncates more than 2% of validation responses; use the next larger candidate.')
!python -m effreason.analyze --run-dirs runs/initial_calibrated_dev --output analysis/calibration/baseline

## Stage 2 - matched accuracy-only smoke training

In [ ]:
!python -m effreason.train --config configs/accuracy.yaml --phase pilot --completion-budget $SELECTED_B --seed 11 --output runs/accuracy_seed11_pilot
# Resume only after an intentional interruption; replace N with the saved checkpoint step.
# !python -m effreason.train --config configs/accuracy.yaml --phase pilot --completion-budget $SELECTED_B --seed 11 --output runs/accuracy_seed11_pilot --resume runs/accuracy_seed11_pilot/checkpoints/checkpoint-N

## Stage 3 - independent gamma pilots
Both adapters start independently from the same pinned base model and use the selected budget.

In [ ]:
!python -m effreason.train --config configs/efficiency.yaml --phase pilot --gamma 0.1 --completion-budget $SELECTED_B --seed 11 --output runs/efficiency_g01_seed11_pilot
!python -m effreason.train --config configs/efficiency.yaml --phase pilot --gamma 0.2 --completion-budget $SELECTED_B --seed 11 --output runs/efficiency_g02_seed11_pilot

## Stage 4 - matched development evaluation

In [ ]:
!python -m effreason.evaluate --config configs/accuracy.yaml --adapter runs/accuracy_seed11_pilot/adapter --condition accuracy --gamma 0 --completion-budget $SELECTED_B --samples-per-example 2 --split dev --output runs/accuracy_seed11_pilot_eval
!python -m effreason.evaluate --config configs/efficiency.yaml --adapter runs/efficiency_g01_seed11_pilot/adapter --condition efficiency_g01 --gamma 0.1 --completion-budget $SELECTED_B --samples-per-example 2 --split dev --output runs/efficiency_g01_seed11_pilot_eval
!python -m effreason.evaluate --config configs/efficiency.yaml --adapter runs/efficiency_g02_seed11_pilot/adapter --condition efficiency_g02 --gamma 0.2 --completion-budget $SELECTED_B --samples-per-example 2 --split dev --output runs/efficiency_g02_seed11_pilot_eval

In [ ]:
!python -m effreason.analyze --run-dirs runs/accuracy_seed11_pilot_eval runs/efficiency_g01_seed11_pilot_eval runs/efficiency_g02_seed11_pilot_eval --training-run-dirs runs/accuracy_seed11_pilot runs/efficiency_g01_seed11_pilot runs/efficiency_g02_seed11_pilot --output analysis/calibration/gamma_comparison

In [ ]:
!python -m effreason.calibrate gamma --reference-run runs/accuracy_seed11_pilot_eval --candidate-runs runs/efficiency_g01_seed11_pilot_eval runs/efficiency_g02_seed11_pilot_eval --output analysis/calibration/gamma_decision.json --accuracy-margin-pp 5 --premise-margin-pp 5 --minimum-token-reduction-percent 15

In [ ]:
with open('analysis/calibration/gamma_decision.json', encoding='utf-8') as stream:
    gamma_decision = json.load(stream)
display(gamma_decision)
if gamma_decision['status'] != 'selected':
    raise RuntimeError('No gamma met the predeclared rule. Do not inspect test data; revise the pilot or report the null selection.')
SELECTED_GAMMA = float(gamma_decision['selected_gamma'])
!python -m effreason.calibrate protocol --budget-decision analysis/calibration/budget_decision.json --gamma-decision analysis/calibration/gamma_decision.json --output analysis/calibration/selected_protocol.yaml
print('Selected gamma:', SELECTED_GAMMA)

## Decision gate
Stop here and record the selected protocol in `experiments.md`. Main steps, seeds, task difficulty, and the nonempty final test set must still be finalized and frozen before any `--phase main --allow-main` command is used.

## Backup all calibration artifacts

In [ ]:
BACKUP = '/content/drive/MyDrive/efficient-reasoning-transfer-backup'  # distinct destination
!mkdir -p $BACKUP
!rsync -a --checksum README.md IMPLEMENTATION_PLAN.md pyproject.toml configs data experiments.md src tests notebooks runs analysis $BACKUP/

All primary artifacts are already under the mounted project directory. The backup cell creates a second copy. Never store credentials in the notebook, and do not describe dependency versions as verified until the GPU save/reload/resume smoke succeeds.